# ESRGAN Astronomy Pipeline

This notebook demonstrates using ESRGAN for super-resolution enhancement of astronomical imagery
(telescope photos, deep-sky objects, planetary surfaces).

The pipeline:
1. Load astronomical imagery (simulated or real)
2. Simulate acquisition noise / downsampling
3. Apply ESRGAN super-resolution
4. Compare with bicubic and evaluate quality
5. Visualize fine detail recovery

**Requirements:** Run from the repository root after `pip install -r requirements.txt`.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.rrdb_net import RRDBNet
from src.inference import run_inference, tiled_upscale
from src.utils import load_image, save_image, compute_psnr, compute_ssim, tensor_to_numpy

import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Simulate Astronomical Image

We generate a synthetic galaxy-like image to demonstrate the pipeline without requiring real data.

In [ ]:
def generate_synthetic_galaxy(size=256, seed=42):
    """Generate a synthetic galaxy-like image."""
    rng = np.random.default_rng(seed)
    img = np.zeros((size, size, 3), dtype=np.float32)

    # Dark background with faint stars
    img += rng.exponential(0.01, (size, size, 3)).clip(0, 0.1)

    # Add scattered stars
    for _ in range(150):
        x, y = rng.integers(0, size, 2)
        brightness = rng.uniform(0.3, 1.0)
        color = rng.choice([[1, 0.9, 0.8], [0.8, 0.9, 1.0], [1, 1, 0.7], [0.9, 0.7, 1.0]])
        r = rng.integers(1, 4)
        y0, y1 = max(0, y-r), min(size, y+r+1)
        x0, x1 = max(0, x-r), min(size, x+r+1)
        img[y0:y1, x0:x1] = np.clip(img[y0:y1, x0:x1] + np.array(color) * brightness, 0, 1)

    # Galaxy core (bright Gaussian blob)
    cx, cy = size // 2, size // 2
    yy, xx = np.mgrid[:size, :size]
    dist = np.sqrt((xx - cx)**2 / (size*0.15)**2 + (yy - cy)**2 / (size*0.08)**2)
    core = np.exp(-dist).clip(0, 1)
    img[:, :, 0] += core * 0.9
    img[:, :, 1] += core * 0.7
    img[:, :, 2] += core * 0.4

    # Spiral arms (rough approximation)
    angles = np.arctan2(yy - cy, xx - cx)
    radii = np.sqrt((xx - cx)**2 + (yy - cy)**2)
    spiral = np.exp(-((radii - size * 0.2) / 20)**2)
    arm1 = spiral * (np.cos(angles * 2 + radii / 20) > 0.7).astype(float)
    arm2 = spiral * (np.cos(angles * 2 + radii / 20 + np.pi) > 0.7).astype(float)
    img[:, :, 0] += arm1 * 0.4 + arm2 * 0.3
    img[:, :, 1] += arm1 * 0.5 + arm2 * 0.4
    img[:, :, 2] += arm1 * 0.6 + arm2 * 0.5

    return np.clip(img, 0, 1)


hr_np = generate_synthetic_galaxy(size=128)
plt.figure(figsize=(6, 6))
plt.imshow(hr_np)
plt.title('Synthetic Astronomical Image (HR Reference)', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'HR image shape: {hr_np.shape}')

## 2. Simulate Telescope Acquisition (Downsample + Noise)

In [ ]:
def simulate_telescope_lr(hr_np, scale=4, noise_std=0.02):
    """Simulate LR acquisition: bicubic downsampling + Gaussian read noise."""
    hr_tensor = torch.from_numpy(hr_np).permute(2, 0, 1).unsqueeze(0)  # (1,C,H,W)
    lr_tensor = F.interpolate(hr_tensor, scale_factor=1/scale, mode='bicubic', align_corners=False)
    lr_tensor = (lr_tensor + torch.randn_like(lr_tensor) * noise_std).clamp(0, 1)
    return lr_tensor.squeeze(0).permute(1, 2, 0).numpy()


lr_np = simulate_telescope_lr(hr_np, scale=4, noise_std=0.015)
bicubic_np = np.array(
    Image.fromarray((lr_np * 255).astype(np.uint8)).resize(
        (hr_np.shape[1], hr_np.shape[0]), Image.BICUBIC
    )
) / 255.0

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(lr_np.clip(0, 1))
axes[0].set_title(f'Simulated LR ({lr_np.shape[1]}×{lr_np.shape[0]})', fontsize=12)
axes[0].axis('off')
axes[1].imshow(bicubic_np.clip(0, 1))
axes[1].set_title(f'Bicubic ×4 ({bicubic_np.shape[1]}×{bicubic_np.shape[0]})', fontsize=12)
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 3. Load ESRGAN Model

In [ ]:
CHECKPOINT_PATH = '../checkpoints/final.pth'

model = RRDBNet(in_nc=3, out_nc=3, nf=64, nb=23, gc=32, scale=4).to(device)
model.eval()

if os.path.isfile(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    state_dict = ckpt.get('generator', ckpt)
    model.load_state_dict(state_dict)
    print(f'Loaded: {CHECKPOINT_PATH}')
else:
    print('No checkpoint — using random weights for pipeline demo.')

## 4. Apply ESRGAN Super-Resolution

In [ ]:
lr_tensor = torch.from_numpy(lr_np.astype(np.float32)).permute(2, 0, 1)

cfg = {
    'model': {'in_nc': 3, 'out_nc': 3, 'nf': 64, 'nb': 23, 'gc': 32, 'scale': 4, 'checkpoint': ''},
    'io': {'input': '', 'output': ''},
    'tiling': {'enabled': True, 'tile_size': 32, 'tile_overlap': 8},
}

sr_tensor = run_inference(model, lr_tensor, cfg, device)
sr_np = tensor_to_numpy(sr_tensor) / 255.0

print(f'LR shape:  {lr_np.shape}')
print(f'SR shape:  {sr_np.shape}')
print(f'HR shape:  {hr_np.shape}')

## 5. Full Comparison

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

images = [
    (lr_np.clip(0, 1),        f'LR Input\n({lr_np.shape[1]}×{lr_np.shape[0]})'),
    (bicubic_np.clip(0, 1),   f'Bicubic ×4\n({bicubic_np.shape[1]}×{bicubic_np.shape[0]})'),
    (sr_np.clip(0, 1),        f'ESRGAN ×4\n({sr_np.shape[1]}×{sr_np.shape[0]})'),
    (hr_np.clip(0, 1),        f'HR Reference\n({hr_np.shape[1]}×{hr_np.shape[0]})'),
]

for ax, (img, title) in zip(axes, images):
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle('Astronomy Pipeline: ESRGAN Super-Resolution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/astronomy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Quality Metrics

In [ ]:
hr_t = torch.from_numpy(hr_np.astype(np.float32)).permute(2, 0, 1)
sr_t = torch.from_numpy(sr_np.astype(np.float32)).permute(2, 0, 1)
bicubic_t = torch.from_numpy(bicubic_np.astype(np.float32)).permute(2, 0, 1)

psnr_bicubic = compute_psnr(bicubic_t, hr_t)
psnr_esrgan  = compute_psnr(sr_t, hr_t)
ssim_bicubic = compute_ssim(bicubic_t, hr_t)
ssim_esrgan  = compute_ssim(sr_t, hr_t)

print('Quality Metrics vs HR Reference')
print('-' * 40)
print(f'Bicubic ×4:  PSNR = {psnr_bicubic:.2f} dB   SSIM = {ssim_bicubic:.4f}')
print(f'ESRGAN  ×4:  PSNR = {psnr_esrgan:.2f} dB   SSIM = {ssim_esrgan:.4f}')
print()
print('Note: After training, ESRGAN should show higher perceptual quality')
print('(sharper details) even if PSNR is similar to bicubic.')

## 7. Detail Zoom — Star Cluster Region

In [ ]:
# Zoom into the top-right quadrant (star cluster area)
H, W = hr_np.shape[:2]
crop = (0, 0, H//2, W//2)  # (y0, x0, y1, x1)
y0, x0, y1, x1 = crop

def zoom_crop(img, y0, x0, y1, x1):
    return img[y0:y1, x0:x1]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
pairs = [
    (zoom_crop(bicubic_np, y0, x0, y1, x1).clip(0, 1), 'Bicubic ×4 (detail)'),
    (zoom_crop(sr_np,      y0, x0, y1, x1).clip(0, 1), 'ESRGAN ×4 (detail)'),
    (zoom_crop(hr_np,      y0, x0, y1, x1).clip(0, 1), 'HR Reference (detail)'),
]
for ax, (img, title) in zip(axes, pairs):
    ax.imshow(img, interpolation='nearest')
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Detail Zoom: Star Field Region', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/astronomy_detail.png', dpi=150, bbox_inches='tight')
plt.show()